In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 1. 엑셀 데이터 불러오기 및 구조 파악
file_path = "모의투자 데이터.xlsx"
df_raw = pd.read_excel(file_path)

years = df_raw.iloc[6].values
quarters = df_raw.iloc[7].values

time_cols = [f"{int(years[i])}_{quarters[i]}" for i in range(7, len(years)) if pd.notna(years[i])]

df_data = df_raw.iloc[8:, :7+len(time_cols)].copy()
df_data.columns = ['코드', '코드명', '결산월', '유형', '아이템코드', '아이템명', '분기'] + time_cols

id_vars = ['코드', '코드명', '아이템명']
df_melted = pd.melt(df_data, id_vars=id_vars, value_vars=time_cols, var_name='시점', value_name='값')
df_melted['값'] = pd.to_numeric(df_melted['값'], errors='coerce')

df_pivoted = df_melted.pivot_table(
    index=['시점', '코드', '코드명'], 
    columns='아이템명', 
    values='값'
).reset_index()

print("Available quarters:", df_pivoted['시점'].unique())

Available quarters: ['2010_1Q' '2010_2Q' '2010_3Q' '2010_4Q' '2011_1Q' '2011_2Q' '2011_3Q'
 '2011_4Q' '2012_1Q' '2012_2Q' '2012_3Q' '2012_4Q' '2013_1Q' '2013_2Q'
 '2013_3Q' '2013_4Q' '2014_1Q' '2014_2Q' '2014_3Q' '2014_4Q' '2015_1Q'
 '2015_2Q' '2015_3Q' '2015_4Q' '2016_1Q' '2016_2Q' '2016_3Q' '2016_4Q'
 '2017_1Q' '2017_2Q' '2017_3Q' '2017_4Q' '2018_1Q' '2018_2Q' '2018_3Q'
 '2018_4Q' '2019_1Q' '2019_2Q' '2019_3Q' '2019_4Q' '2020_1Q' '2020_2Q'
 '2020_3Q' '2020_4Q' '2021_1Q' '2021_2Q' '2021_3Q' '2021_4Q' '2022_1Q'
 '2022_2Q' '2022_3Q' '2022_4Q' '2023_1Q' '2023_2Q' '2023_3Q' '2023_4Q'
 '2024_1Q' '2024_2Q' '2024_3Q' '2024_4Q' '2025_1Q' '2025_2Q' '2025_3Q'
 '2025_4Q' '2026_1Q' '2026_2Q' '2026_3Q' '2026_4Q']


In [7]:
# 금리 인상기 각 구간별 분기 매핑 및 랭킹 로직 테스트
# 구간 1: 2010년 7월 ~ 2011년 6월 -> 2010_2Q ~ 2011_2Q (또는 기준이 되는 발표 시점 2010_2Q/2010_3Q)
# 금리 인상기 시작 시점 재무제표 기준으로 종목 스크리닝 및 종합 순위 산출

periods_map = {
    "1차 금리인상기 (2010.07 ~ 2011.06)": "2010_2Q",
    "2차 금리인상기 (2017.11 ~ 2018.11)": "2017_3Q",
    "3차 금리인상기 (2021.08 ~ 2023.01)": "2021_2Q",
    "4차 금리인상기 (2026.07 ~ 현재)": "2026_2Q"
}

results = {}

for period_name, qtr in periods_map.items():
    df_qtr = df_pivoted[df_pivoted['시점'] == qtr].copy()
    
    # 조건 1: 부채비율 < 100%
    # 조건 2: 이자보상배율 > 3배
    filtered = df_qtr[(df_qtr['부채비율(%)'] < 100) & (df_qtr['이자보상배율(배)'] > 3)].copy()
    
    # 조건 3: PBR 낮을수록 좋음 (오름차순) - 단, 양수 기준
    filtered = filtered[filtered['P/B(배)'] > 0]
    filtered['rank_PBR'] = filtered['P/B(배)'].rank(ascending=True, method='min')
    
    # 조건 4: EV/EBITDA 낮을수록 좋음 (오름차순) - 단, 양수 기준
    filtered = filtered[filtered['EV/EBITDA(배)'] > 0]
    filtered['rank_EV_EBITDA'] = filtered['EV/EBITDA(배)'].rank(ascending=True, method='min')
    
    # 조건 5: FCF 높을수록 좋음 (내림차순)
    filtered['rank_FCF'] = filtered['FCF(천원)'].rank(ascending=False, method='min')
    
    # 종합 순위 (단순 가중 평균)
    filtered['avg_rank'] = (filtered['rank_PBR'] + filtered['rank_EV_EBITDA'] + filtered['rank_FCF']) / 3.0
    filtered['final_rank'] = filtered['avg_rank'].rank(ascending=True, method='min')
    
    top10 = filtered.sort_values(by='final_rank').head(20)
    results[period_name] = top10[['final_rank', '코드', '코드명', 'P/B(배)', 'EV/EBITDA(배)', 'FCF(천원)', '부채비율(%)', '이자보상배율(배)']]

for p_name, top_df in results.items():
    print(f"\n================ {p_name} (기준 분기: {periods_map[p_name]}) Top 10 ===============")
    print(top_df.to_string(index=False))


================ 1차 금리인상기 (2010.07 ~ 2011.06) (기준 분기: 2010_2Q) Top 10 ===============
 final_rank      코드      코드명  P/B(배)  EV/EBITDA(배)      FCF(천원)  부채비율(%)  이자보상배율(배)
        1.0 A003240     태광산업    0.34          4.03  53864253.93    24.42     314.99
        2.0 A058650    세아홀딩스    0.31          7.61  57614317.90    20.85      30.48
        3.0 A006200  한국전자홀딩스    0.27          7.43   7712503.39     6.49    2683.28
        4.0 A000860   강남제비스코    0.35          8.75   9221521.54    28.22     176.50
        4.0 A026940     부국철강    0.46          5.63  13020930.28    35.71     184.64
        6.0 A104700     한국철강    0.40         10.04   9360258.87    39.43      71.45
        7.0 A000320    노루홀딩스    0.28         14.45  15774574.61    40.74       8.31
        8.0 A003480 한진중공업홀딩스    0.29         16.69  29141162.24     3.82      80.76
        8.0 A009300     삼아제약    0.36         12.22   6194062.04    41.13       6.31
       10.0 A016590    신대양제지    0.36         11.51   4380114.95    42.56 